In [1]:
from pyspark.sql.functions import (
    col, year, month, dayofmonth, weekofyear, date_format,
    weekday, when, expr, to_date, row_number
)
from pyspark.sql.types import StructType, StructField, StringType,IntegerType
import ConnectionConfig as cc
cc.setupEnvironment()

Environment variables are set...


In [2]:
#config
cc.setupEnvironment()
print(cc.config.sections())

Environment variables are set...
['default', 'tutorial_op', 'catchem', 'kafka']


In [3]:
spark = cc.startLocalCluster("DIM_RAIN")
spark.getActiveSession()

In [4]:
#make connection
cc.config.read('config.ini')
cc.set_connectionProfile("catchem")

In [5]:
# ---- 1. Schema definiëren ----
rain_schema = StructType([
    StructField("RainSurKey", IntegerType(), False),
    StructField("RainCode", StringType(), False),
    StructField("RainDescription", StringType(), False)
])

In [6]:
rain_data = [
    (1, "RAIN", "Weer met regen (codes 200-699)"),
    (2, "NORAIN", "Weer zonder regen"),
    (3, "UNKNOWN", "Regen situatie onbekend")
]

In [7]:
# ---- 3. DataFrame aanmaken ----
rain_dim_df = spark.createDataFrame(rain_data, schema=rain_schema)

# ---- 4. Controleren ----
rain_dim_df.show(truncate=False)

+----------+--------+------------------------------+
|RainSurKey|RainCode|RainDescription               |
+----------+--------+------------------------------+
|1         |RAIN    |Weer met regen (codes 200-699)|
|2         |NORAIN  |Weer zonder regen             |
|3         |UNKNOWN |Regen situatie onbekend       |
+----------+--------+------------------------------+



In [9]:
rain_dim_df.coalesce(1).write.format("delta").mode("overwrite").save("delta/RAIN_DIM")
